# Contextual versus isolated detail representations, by age

For each detail, the **contextualization shift** is `1 - cos(contextual vector, isolated vector)`: how differently GPT-OSS
represents the detail inside its text than when encoded alone. Both vectors are mean-pooled, at every stored layer.

This notebook follows the shift against

- **source age**: the age the story was written for
- **target age**: the age a summary was written for (younger, actual, older summaries of the same story)

for **original details** and for the **summary details** of each of the three summaries, and then relates it to the
span-search compression of each story. It complements `gpt_oss_detail_representation_alignment.ipynb`, which covers
the actual-age summary only, at the final layer.

Input: `contextualization_shift/details.csv` from `hpc/analysis/submit_span_search.sh`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import linregress, spearmanr, t as student_t

RESULTS = Path(os.environ.get("RESULTS_DIR", "results"))
SHIFT = RESULTS / "analysis/representation/detail_compression/contextualization_shift/details.csv"
SPAN = RESULTS / "analysis/representation/detail_compression/span_search/contextual_mean/summaries.csv"
LAYER = 12          # layer for the main figures (nearest stored layer is used)
AGE_BIN = 1.0
KINDS = ["original", "younger", "actual", "older"]     # details of the original text, then of each summary
COLORS = {"original": "#52514e", "younger": "#2a78d6", "actual": "#eb6834", "older": "#1baf7a"}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": MUTED,
    "axes.labelcolor": "#52514e", "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 110, "font.size": 10,
})

raw = pd.read_csv(SHIFT)
raw["kind"] = np.where(raw.record_type.eq("original"), "original", raw.condition)
raw = raw[raw.kind.isin(KINDS)]
layers = sorted(raw.layer.unique())
layer = layers[int(np.argmin(np.abs(np.asarray(layers) - LAYER)))]
print(f"stored layers {layers}; main layer {layer}")
print(raw.groupby("kind").agg(details=("shift", "size"), stories=("story_key", "nunique")))

In [ ]:
# One row per record and layer: the story is the unit, never the individual detail.
rec = (raw.groupby(["story_key", "kind", "record_id", "source_age", "target_age", "layer"], dropna=False)
          .agg(shift=("shift", "mean"), details=("shift", "size")).reset_index())
rec["target_age"] = rec.target_age.fillna(rec.source_age)
rec["age_gap"] = rec.target_age - rec.source_age         # how far the summary's target age is from the story's age
rec["age_bin"] = np.floor(rec.source_age / AGE_BIN) * AGE_BIN
main = rec[rec.layer.eq(layer)]

def binned(frame, y, x="age_bin", n_boot=400, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for (kind, bin_), g in frame.groupby(["kind", x]):
        values = g[y].to_numpy()
        boots = rng.choice(values, size=(n_boot, len(values))).mean(axis=1)
        rows.append({"kind": kind, x: bin_, "mean": values.mean(), "lo": np.percentile(boots, 2.5), "hi": np.percentile(boots, 97.5), "n": len(values)})
    return pd.DataFrame(rows)

def slope(g, y="shift", x="source_age"):
    g = g.dropna(subset=[x, y])
    fit = linregress(g[x], g[y])
    half = student_t.ppf(0.975, len(g) - 2) * fit.stderr
    return fit.slope, fit.slope - half, fit.slope + half, fit.pvalue, len(g)

## 1. Shift versus source age, for original and summary details
Story-level mean shift at the main layer. Each line is one kind of text: the original, and the younger, actual and older summary of the same stories.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.4))
for kind in KINDS:
    sub = main[main.kind.eq(kind)]
    if sub.empty:
        continue
    ax.scatter(sub.source_age, sub["shift"], s=7, alpha=0.15, color=COLORS[kind], linewidths=0)
    line = binned(sub, "shift").sort_values("age_bin")
    x = line.age_bin + AGE_BIN / 2
    ax.fill_between(x, line.lo, line.hi, color=COLORS[kind], alpha=0.13, linewidth=0)
    ax.plot(x, line["mean"], color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=kind)
ax.set_xlabel("Source age (years)"); ax.set_ylabel("Mean shift, 1 - cos(contextual, isolated)")
ax.set_title(f"Contextualization shift by source age (layer {layer})", loc="left", fontsize=11)
ax.legend(frameon=False, ncol=4, loc="upper right")
fig.tight_layout();

In [ ]:
rows = []
for kind in KINDS:
    g = main[main.kind.eq(kind)]
    if g.story_key.nunique() >= 4:
        s, lo, hi, p, n = slope(g)
        rows.append({"kind": kind, "stories": n, "slope per year": s, "ci low": lo, "ci high": hi, "p": p})
pd.DataFrame(rows).round(4)

## 2. Shift versus target age
For the summaries only: does asking for a younger or older reader change how context-dependent the summary's details are? Left, by summary type at each source age. Right, against the gap between target and source age.

In [ ]:
summ = main[main.kind.isin(["younger", "actual", "older"])]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
for kind in ["younger", "actual", "older"]:
    sub = summ[summ.kind.eq(kind)]
    line = binned(sub, "shift").sort_values("age_bin")
    x = line.age_bin + AGE_BIN / 2
    ax.fill_between(x, line.lo, line.hi, color=COLORS[kind], alpha=0.13, linewidth=0)
    ax.plot(x, line["mean"], color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=kind)
ax.set_xlabel("Source age (years)"); ax.set_ylabel("Mean shift of summary details"); ax.legend(frameon=False, title="summary")
ax = axes[1]
for kind in ["younger", "actual", "older"]:
    sub = summ[summ.kind.eq(kind)]
    ax.scatter(sub.age_gap + np.random.default_rng(1).normal(0, 0.05, len(sub)), sub["shift"], s=9, alpha=0.4, color=COLORS[kind], linewidths=0)
    ax.plot(sub.age_gap.mean(), sub["shift"].mean(), "o", color=COLORS[kind], mfc="#fcfcfb", mew=2, ms=9)
ax.set_xlabel("Target age minus source age (years)"); ax.set_ylabel("Mean shift of summary details")
fig.tight_layout();

## 3. Paired within story: summary details minus original details
Positive means the summary's details depend on their context more than the original's details do.

In [ ]:
wide = main.pivot_table(index=["story_key", "source_age"], columns="kind", values="shift").reset_index()
fig, ax = plt.subplots(figsize=(7.5, 4))
for kind in ["younger", "actual", "older"]:
    if kind not in wide:
        continue
    diff = (wide[kind] - wide["original"]).dropna()
    ages = wide.loc[diff.index, "source_age"]
    ax.scatter(ages, diff, s=9, alpha=0.3, color=COLORS[kind], linewidths=0)
    frame = pd.DataFrame({"kind": kind, "age_bin": np.floor(ages / AGE_BIN) * AGE_BIN, "d": diff})
    line = binned(frame, "d").sort_values("age_bin")
    ax.plot(line.age_bin + AGE_BIN / 2, line["mean"], color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=f"{kind} - original")
ax.axhline(0, color=MUTED, lw=1)
ax.set_xlabel("Source age (years)"); ax.set_ylabel("Difference in mean shift"); ax.legend(frameon=False)
fig.tight_layout();

## 4. Across layers
Left: average shift at each layer. Right: how steeply the shift changes with source age at each layer. Early layers carry little context, so large shifts there mean the two encodings differ in surface form (position, framing) rather than meaning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for kind in KINDS:
    g = rec[rec.kind.eq(kind)]
    by_layer = g.groupby("layer")["shift"].mean()
    axes[0].plot(by_layer.index, by_layer.values, color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=kind)
    slopes = {l: slope(gl)[0] for l, gl in g.groupby("layer") if gl.story_key.nunique() >= 4}
    axes[1].plot(list(slopes), list(slopes.values()), color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=kind)
axes[0].set_xlabel("Layer"); axes[0].set_ylabel("Mean shift"); axes[0].legend(frameon=False)
axes[1].axhline(0, color=MUTED, lw=1); axes[1].set_xlabel("Layer"); axes[1].set_ylabel("Change in shift per year of source age")
fig.tight_layout();

## 5. Relation to span-search compression
Per story and summary: does a larger contextualization shift go with each summary detail standing for more of the story? Both measures change with age, so the right panel compares the parts of each that age does not explain.

In [ ]:
if not SPAN.exists():
    print(f"Span-search results not found at {SPAN}; run the span search first.")
else:
    span = pd.read_csv(SPAN)
    span_layer = span.layer.unique()[int(np.argmin(np.abs(span.layer.unique() - layer)))]
    span = span[span.layer.eq(span_layer) & span.length_penalty.eq(span.length_penalty.unique()[np.argmin(np.abs(span.length_penalty.unique() - 0.1))])]
    link = (main[main.kind.isin(["younger", "actual", "older"])]
            .merge(span[["story_key", "condition", "mean_span_len"]], left_on=["story_key", "kind"], right_on=["story_key", "condition"]))
    print(f"{len(link)} story-summaries matched; span layer {span_layer}")
    def residual(y, x):
        return y - np.polyval(np.polyfit(x, y, 1), x)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for kind in ["younger", "actual", "older"]:
        g = link[link.kind.eq(kind)]
        axes[0].scatter(g["shift"], g.mean_span_len, s=12, alpha=0.6, color=COLORS[kind], linewidths=0, label=kind)
        axes[1].scatter(residual(g["shift"].to_numpy(), g.source_age.to_numpy()), residual(g.mean_span_len.to_numpy(), g.source_age.to_numpy()),
                        s=12, alpha=0.6, color=COLORS[kind], linewidths=0)
    rho = spearmanr(link["shift"], link.mean_span_len).statistic
    axes[0].set_title(f"Raw (Spearman {rho:.2f})", loc="left", fontsize=10)
    axes[0].set_xlabel("Mean contextualization shift"); axes[0].set_ylabel("Mean span length (source details per summary detail)"); axes[0].legend(frameon=False)
    rs, rl = residual(link["shift"].to_numpy(), link.source_age.to_numpy()), residual(link.mean_span_len.to_numpy(), link.source_age.to_numpy())
    axes[1].set_title(f"After removing the source-age trend (Spearman {spearmanr(rs, rl).statistic:.2f})", loc="left", fontsize=10)
    axes[1].set_xlabel("Shift, residual"); axes[1].set_ylabel("Span length, residual")
    fig.tight_layout()

## Reading these results

- Shift compares two different inputs (the detail in its text, and the detail alone), so part of it is always present. Look at how it changes with age and between summary types, not at its absolute size.
- The unit is the story or summary; individual details from one text are not independent.
- Younger summaries exist for few stories, so that line has the widest intervals.
- Mean pooling is used for both vectors, so the two are read out the same way.